# Ontologies from Zero: Semantic Guardrails for Agentic Systems (Solution)
This notebook provides the fully implemented dual-gate validation pipeline.

In [ ]:
!pip install pydantic rdflib pyshacl

## Step 1: Gate 1 - Structural Validation (Pydantic)

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class ToolExecutionPayload(BaseModel):
    agent_id: str = Field(..., description="Unique identifier of the executing agent")
    action_type: str = Field(..., description="Type of action requested")
    target_resource: str = Field(..., description="Target resource or account")
    amount: float = Field(..., ge=0.0, description="Transaction or operation amount")

# Test payload
sample = ToolExecutionPayload(agent_id="agent_007", action_type="transfer", target_resource="account_42", amount=150.0)
print("Gate 1 Passed:", sample.model_dump())

## Step 2: Gate 2 - Semantic Validation (RDFLib & SHACL)

In [ ]:
from rdflib import Graph, Namespace, Literal, RDF, URIRef, XSD
from pyshacl import validate

EX = Namespace("http://example.org/agent-ns#")
SH = Namespace("http://www.w3.org/ns/shacl#")

# Populate Data Graph with the proposed action instance
data_graph = Graph()
action_uri = EX["Execution_101"]
data_graph.add((action_uri, RDF.type, EX.FinancialAction))
data_graph.add((action_uri, EX.agentId, Literal("agent_007")))
data_graph.add((action_uri, EX.amount, Literal(150.0, datatype=XSD.float)))
data_graph.add((action_uri, EX.hasApproval, Literal(False, datatype=XSD.boolean)))

print("Data graph initialized with", len(data_graph), "triples.")

In [ ]:
# Populate SHACL Shapes Graph for business rules
shapes_graph = Graph()
shape_uri = EX["FinancialActionShape"]

# SHACL rule: If amount > 100.0, must have approval = true
# Expressed as: amount <= 100.0  OR  hasApproval = true
shapes_ttl = """
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix ex: <http://example.org/agent-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:FinancialActionShape
    a sh:NodeShape ;
    sh:targetClass ex:FinancialAction ;
    sh:or (
        [ sh:property [ sh:path ex:amount ; sh:maxInclusive 100.0 ] ]
        [ sh:property [ sh:path ex:hasApproval ; sh:hasValue true ] ]
    ) ;
    sh:message "Transactions over 100.0 require explicit manager approval and cannot pass automatically." .
"""

shapes_graph.parse(data=shapes_ttl, format="turtle")

# Run Validation
conforms, results_graph, results_text = validate(
    data_graph,
    shacl_graph=shapes_graph,
    inference='rdfs',
    abort_on_first=False,
    meta_shacl=False,
    advanced_features=True,
    js=False,
    serialization='turtle'
)

print("Gate 2 Conforms:", conforms)
print("Validation Report:\n", results_text)

## Step 3: Same Action, Now With Manager Approval
Flip `hasApproval` to `true` and re-run Gate 2. The 150.0 transfer should now conform.

In [ ]:
# Replace the approval flag on the same action
data_graph.set((action_uri, EX.hasApproval, Literal(True, datatype=XSD.boolean)))

conforms, results_graph, results_text = validate(
    data_graph,
    shacl_graph=shapes_graph,
    inference='rdfs',
    advanced_features=True,
)

print("Gate 2 Conforms (with approval):", conforms)